In [38]:
import pandas as pd
import warnings
import sys
import os
import pickle
import numpy as np

sys.path.append(os.path.abspath(".."))

from src.app.core.api import Features
from src.app.core.model import Model

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

In [39]:
DATA_PATH = "../hw6/data/val_features.csv"

MODEL_PATH = "../hw6/models/logistic_regression.pkl"

SCALER_PATH = "../hw6/models/logistic_regression_scaler.pkl"

IDX_TO_COMPARE_COUNT = 1000

In [40]:
df_val = pd.read_csv(DATA_PATH)
print(f"Валидационная выборка: {df_val.shape}")


Валидационная выборка: (52656, 10)


In [41]:
features = Features.from_dataframe(df_val)

Логистическая регрессия была обучена на масштабированных признаках

In [42]:
with open(SCALER_PATH, "rb") as f:
    scaler = pickle.load(f)

df_scaled = scaler.transform(df_val)

Обертка

In [43]:
wrapper = Model(MODEL_PATH, SCALER_PATH)

Модель

In [44]:
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

Проверяем пробы на обертке и модели

In [45]:
probas_wrapper = wrapper.predict_proba_batch(df_val[:IDX_TO_COMPARE_COUNT])

In [46]:
probas_model = model.predict_proba(df_scaled[:IDX_TO_COMPARE_COUNT])[:, 1]

In [47]:
if np.allclose(probas_wrapper, probas_model, atol=1e-8):
    print("Все вероятности совпадают")
else:
    print("Вероятности не совпадают")

Все вероятности совпадают


Проверка калькулятора

In [50]:
idx = 18

client_data = df_val.iloc[idx : idx + 1]

features = Features.from_dataframe(client_data)

result = wrapper.get_scoring_result(features)

print("Информация о клиенте")
print(client_data)

print("Результат")
print(f"Решение о выдаче займа: {result.decision}")
print(f"Сумма: {result.amount}")
print(f"threshold: {result.threshold}")
print(f"proba: {result.proba:.8f}")
print(f"proba модели напрямую: {model.predict_proba(df_scaled[idx : idx + 1])[:, 1]}")

Информация о клиенте
    weighted_ext_score  ext_source_3  ext_source_2  days_registration  days_birth  days_id_publish  annuity_to_income_proportion  interest_rate  days_employed  amt_annuity
18            0.437571      0.698667      0.569975            -3377.0      -21040            -4385                      0.333333      21.314007          -2228     225000.0
Результат
Решение о выдаче займа: ScoringDecision.ACCEPTED
Сумма: 302400
threshold: 0.3
proba: 0.14288944
proba модели напрямую: [0.14288944]


Вывод для n заявлений

In [49]:
n = 1000

results = []

for idx in range(n):
    features = Features.from_dataframe(df_val.iloc[idx : idx + 1])
    result = wrapper.get_scoring_result(features)

    results.append(
        {
            "proba": result.proba,
            "weighted_ext_score": features.weighted_ext_score,
            "ext_source_3": features.ext_source_3,
            "ext_source_2": features.ext_source_2,
            "days_registration": features.days_registration,
            "days_birth": features.days_birth,
            "days_id_publish": features.days_id_publish,
            "annuity_to_income_proportion": features.annuity_to_income_proportion,
            "interest_rate": features.interest_rate,
            "days_employed": features.days_employed,
            "amt_annuity": features.amt_annuity,
        }
    )

results = pd.DataFrame(results)

results.to_csv(f"../{n}_заявлений.csv")

results

,proba,weighted_ext_score,ext_source_3,ext_source_2,days_registration,days_birth,days_id_publish,annuity_to_income_proportion,interest_rate,days_employed,amt_annuity
0,0.332764,0.336558,0.510089,0.468826,-13252.0,-24248,-4638,0.130267,10.286383,365243,17586.0
1,0.164767,0.656764,0.812823,0.585148,-2832.0,-13436,-5901,0.084543,5.262763,-193,13315.5
2,0.209276,0.660741,0.665855,0.415780,-9257.0,-20066,-3474,0.185457,16.031722,-153,29209.5
3,0.666126,0.225514,0.535276,0.266520,-4957.0,-16953,-510,0.209240,0.926304,-4400,23539.5
4,0.251549,0.503131,0.792264,0.667729,-3208.0,-9028,-1695,0.165450,2.384518,-1056,44671.5
...,...,...,...,...,...,...,...,...,...,...,...
995,0.420819,0.304658,0.385915,0.508934,-6142.0,-20668,-4173,0.055556,10.620976,365243,6750.0
996,0.160419,0.501771,0.768808,0.689692,-149.0,-22652,-3972,0.182171,4.859400,365243,28692.0
997,0.607638,0.303725,0.331251,0.567003,-2429.0,-14499,-2447,0.292133,5.308712,-2482,39438.0
998,0.519435,0.444344,0.408359,0.458267,-4290.0,-10396,-3057,0.179771,6.537401,-204,38830.5
